In [ ]:
!pip install -qU "transformers>=4.43" "accelerate>=0.33" "datasets>=2.20" "evaluate>=0.4" "scikit-learn>=1.3" "pyarrow" "pandas>=2.2" "matplotlib>=3.8" "openpyxl"
!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

import os, math, json, random, re, itertools, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

from torch.utils.data import Dataset
from datasets import Dataset as HFDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    set_seed,
)

import torch
torch.backends.cuda.sdp_kernel(enable_flash=False, enable_mem_efficient=True, enable_math=False)


print("Torch:", torch.__version__)
import transformers, accelerate, datasets
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 119.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 130.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you ha

/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


In [ ]:
CFG = {
  "seed": 42,
  "model_id": "AI-Sweden-Models/gpt-sw3-6.7b-v2-instruct",
  "excel_path": "radiologue dataset.xlsx",
  "text_col": "Summa",
  "label_col": "End Result",
  "min_samples_per_class": 50,
  "max_length": 1024,
  "epochs": 2,
  "lr": 1.5e-5,
  "warmup_ratio": 0.05,
  "weight_decay": 0.01,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "logging_steps": 20,
  "eval_steps": 20,
  "save_steps": 20,
  "save_total_limit": 2,
  "bf16": True,
  "fp16": False,
  "gradient_checkpointing": True,
  "attn_impl": "sdpa",
  "num_beams": 1,
  "gen_max_new_tokens": 8,
  "output_dir": "gptsw3_radiology_cls_full",
}
os.makedirs(CFG["output_dir"], exist_ok=True)
set_seed(CFG["seed"])

In [ ]:
df = pd.read_excel(CFG["excel_path"])
df = df[[CFG["text_col"], CFG["label_col"]]].dropna().copy()
df[CFG["text_col"]] = df[CFG["text_col"]].astype(str).str.strip()
df[CFG["label_col"]] = df[CFG["label_col"]].astype(str).str.strip()
df = df[(df[CFG["text_col"]].str.len() > 0) & (df[CFG["label_col"]].str.len() > 0)]
counts = df[CFG["label_col"]].value_counts()
keep_labels = counts[counts >= CFG["min_samples_per_class"]].index.tolist()
print(f"Original samples: {len(df)} | Unique labels: {df[CFG['label_col']].nunique()}")
print(f"Keeping labels with >= {CFG['min_samples_per_class']} samples: {len(keep_labels)} labels")
df = df[df[CFG["label_col"]].isin(keep_labels)].reset_index(drop=True)
print(f"Filtered samples: {len(df)}")
df = df.drop_duplicates(subset=[CFG["text_col"], CFG["label_col"]]).reset_index(drop=True)
train_df, test_df = train_test_split(
    df, test_size=0.15, random_state=CFG["seed"], stratify=df[CFG["label_col"]]
)
train_df, val_df = train_test_split(
    train_df, test_size=0.15, random_state=CFG["seed"], stratify=train_df[CFG["label_col"]]
)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Original samples: 1353 | Unique labels: 33
Keeping labels with >= 50 samples: 14 labels
Filtered samples: 1135
Train: 818, Val: 145, Test: 171


In [ ]:
label_list = sorted(sorted(keep_labels), key=lambda x: x.lower())
label_to_id = {lbl: i for i, lbl in enumerate(label_list)}
id_to_label = {i: lbl for lbl, i in label_to_id.items()}
def build_label_block(labels: List[str]) -> str:
    return " | ".join(labels)
LABEL_BLOCK = build_label_block(label_list)
SWEDISH_SYSTEM_INSTR = (
    "Klassificera den kliniska texten till exakt en etikett från listan över radiologiska protokoll (End Result).\n"
    "Svara ENDAST med etiketten, inget annat."
)
def make_prompt(clinical_text: str) -> str:
    return (
        f"{SWEDISH_SYSTEM_INSTR}\n\n"
        f"Tillåtna etiketter: {LABEL_BLOCK}\n\n"
        f"Text: {clinical_text.strip()}\n\n"
        f"Svar:"
    )
print("Exempel prompt:\n", make_prompt("Patient med buksmärta sedan två dagar...")[:400], "...")

Exempel prompt:
 Klassificera den kliniska texten till exakt en etikett från listan över radiologiska protokoll (End Result).
Svara ENDAST med etiketten, inget annat.

Tillåtna etiketter: A1 | A3 | A4 | A4 med sen fas | B10 | B2 | B3 | BBA1 | L5 | L7 | N1 | N14 | N4 | P2

Text: Patient med buksmärta sedan två dagar...

Svar: ...


In [ ]:
from huggingface_hub import login
login("")

In [ ]:
from typing import Dict
tokenizer = AutoTokenizer.from_pretrained(CFG["model_id"], use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
def encode_row(text: str, label: str, tokenizer, max_length: int) -> Dict[str, torch.Tensor]:
    prompt = make_prompt(text)
    answer = " " + label.strip()
    prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
    answer_ids = tokenizer(answer + tokenizer.eos_token, add_special_tokens=False).input_ids
    total = len(prompt_ids) + len(answer_ids)
    if total > max_length:
        excess = total - max_length
        prompt_ids = prompt_ids[excess:]
    input_ids = prompt_ids + answer_ids
    attention_mask = [1] * len(input_ids)
    labels = [-100] * len(prompt_ids) + answer_ids.copy()
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}
class CausalLMDataset(Dataset):
    def __init__(self, df: pd.DataFrame, text_col: str, label_col: str, tokenizer, max_length: int):
        self.df = df.reset_index(drop=True)
        self.text_col = text_col
        self.label_col = label_col
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = encode_row(row[self.text_col], row[self.label_col], self.tokenizer, self.max_length)
        return enc

import torch
def collate(batch):
    pad_id = tokenizer.pad_token_id
    max_len = max(len(x["input_ids"]) for x in batch)

    input_ids = []
    attention_mask = []
    labels = []

    for x in batch:
        L = len(x["input_ids"])
        pad = max_len - L

        input_ids.append(x["input_ids"] + [pad_id] * pad)
        attention_mask.append(x["attention_mask"] + [0] * pad)
        labels.append(x["labels"] + [-100] * pad)   # VERY IMPORTANT: pad labels with -100

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }
train_ds = CausalLMDataset(train_df, CFG["text_col"], CFG["label_col"], tokenizer, CFG["max_length"])
val_ds   = CausalLMDataset(val_df,   CFG["text_col"], CFG["label_col"], tokenizer, CFG["max_length"])
test_ds  = CausalLMDataset(test_df,  CFG["text_col"], CFG["label_col"], tokenizer, CFG["max_length"])
print("Example keys:", collate([train_ds[0], train_ds[1]]).keys())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

Example keys: dict_keys(['input_ids', 'attention_mask', 'labels'])


In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
attn_impl = CFG["attn_impl"]
dtype = torch.bfloat16 if CFG["bf16"] else (torch.float16 if CFG["fp16"] else torch.float32)
model = AutoModelForCausalLM.from_pretrained(
    CFG["model_id"],
    torch_dtype=dtype,
    attn_implementation=attn_impl if attn_impl in ("flash_attention_2", "sdpa", "eager") else "eager",
    device_map="auto",
)
model.gradient_checkpointing_enable()
model.config.use_cache = False
print(f"Loaded {CFG['model_id']}")

config.json:   0%|          | 0.00/982 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/9.99G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/9.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/8.06G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded AI-Sweden-Models/gpt-sw3-6.7b-v2-instruct


In [ ]:
from transformers import TrainingArguments, Trainer
args = TrainingArguments(
    output_dir=CFG["output_dir"],
    num_train_epochs=CFG["epochs"],
    learning_rate=CFG["lr"],
    warmup_ratio=CFG["warmup_ratio"],
    weight_decay=CFG["weight_decay"],
    per_device_train_batch_size=CFG["per_device_train_batch_size"],
    per_device_eval_batch_size=CFG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CFG["gradient_accumulation_steps"],
    bf16=CFG["bf16"],
    fp16=CFG["fp16"],
    gradient_checkpointing=CFG["gradient_checkpointing"],
    logging_steps=CFG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CFG["eval_steps"],
    save_steps=CFG["save_steps"],
    save_total_limit=CFG["save_total_limit"],
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    report_to="none",
)
trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=args,
    data_collator=collate,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)
print("Starting training …")
train_out = trainer.train()
print("Training done.")
trainer.save_model(CFG["output_dir"])
tokenizer.save_pretrained(CFG["output_dir"])

/tmp/ipython-input-747702212.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 3, 'bos_token_id': 2}.


Starting training …


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
20,1.140300,0.573873
40,0.437800,0.228727
60,0.200000,0.113987
80,0.162500,0.131009
100,0.155100,0.081778
120,0.089100,0.086852
140,0.044700,0.080598
160,0.056200,0.095796
180,0.082300,0.091879
200,0.056100,0.090310


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 2048}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Training done.


('gptsw3_radiology_cls_full/tokenizer_config.json',
 'gptsw3_radiology_cls_full/special_tokens_map.json',
 'gptsw3_radiology_cls_full/chat_template.jinja',
 'gptsw3_radiology_cls_full/spiece.model',
 'gptsw3_radiology_cls_full/added_tokens.json')

In [ ]:
import re, torch
from typing import List, Optional
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip().lower())

norm_label_map = {normalize(lbl): lbl for lbl in label_list}

def closest_label(pred: str) -> Optional[str]:
    p = normalize(pred)
    if p in norm_label_map:
        return norm_label_map[p]
    for lbl in label_list:
        n = normalize(lbl)
        if p.startswith(n) or n.startswith(p):
            return lbl
    try:
        import difflib
        match = difflib.get_close_matches(p, list(norm_label_map.keys()), n=1, cutoff=0.0)
        if match:
            return norm_label_map[match[0]]
    except Exception:
        pass
    return None

def _extract_first_label_from_text(decoded: str) -> str:
    after = decoded.split("Svar:")[-1].strip()
    return re.split(r"[\n\r]|[.，,;:!?\u3002]", after, maxsplit=1)[0].strip()

@torch.no_grad()
def generate_label(texts: List[str], model, tokenizer, max_new_tokens=CFG["gen_max_new_tokens"]) -> List[str]:
    """Top-1 (greedy/beam=1) prediction used for standard metrics."""
    model.eval()
    outs = []
    for t in texts:
        prompt = make_prompt(t)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=CFG["max_length"]).to(model.device)
        gen = model.generate(
            **inputs,
            do_sample=False,
            num_beams=max(1, CFG["num_beams"]),
            num_return_sequences=1,
            max_new_tokens=max_new_tokens,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
        full = tokenizer.decode(gen[0], skip_special_tokens=True)
        pred_raw = _extract_first_label_from_text(full)
        outs.append(closest_label(pred_raw) or pred_raw)
    return outs

@torch.no_grad()
def generate_topk_labels(texts: List[str], model, tokenizer, k: int = 3, max_new_tokens=CFG["gen_max_new_tokens"]) -> List[List[str]]:
    """
    Top-k via deterministic beam search (no sampling).
    Returns a list of up to k mapped labels (deduped, order-preserving) per input.
    """
    model.eval()
    all_outs = []
    beams = max(k, 2)
    for t in texts:
        prompt = make_prompt(t)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=CFG["max_length"]).to(model.device)
        gen = model.generate(
            **inputs,
            do_sample=False,
            num_beams=beams,
            num_return_sequences=min(k, beams),
            max_new_tokens=max_new_tokens,
            early_stopping=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
        cand_labels = []
        seen = set()
        for seq in gen:
            full = tokenizer.decode(seq, skip_special_tokens=True)
            pred_raw = _extract_first_label_from_text(full)
            mapped = (closest_label(pred_raw) or pred_raw)
            if mapped in norm_label_map.values() and mapped not in seen:
                cand_labels.append(mapped)
                seen.add(mapped)
            if len(cand_labels) >= k:
                break
        all_outs.append(cand_labels)
    return all_outs

def eval_split(df_split: pd.DataFrame, name: str, topk: int = 3):
    y_true = df_split[CFG["label_col"]].tolist()

    y_pred = generate_label(df_split[CFG["text_col"]].tolist(), model, tokenizer)
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    topk_preds = generate_topk_labels(df_split[CFG["text_col"]].tolist(), model, tokenizer, k=topk)
    topk_hits = [(yt in preds) for yt, preds in zip(y_true, topk_preds)]
    acc_topk = sum(topk_hits) / len(topk_hits) if topk_hits else 0.0

    print(f"\n=== {name} ===")
    print(f"Top-1  -> Accuracy: {acc:.4f} | F1-macro: {f1_macro:.4f} | F1-weighted: {f1_weighted:.4f}")
    print(f"Top-{topk} -> Accuracy: {acc_topk:.4f}")
    print("\nClassification report (Top-1):\n")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))
    return y_true, y_pred, topk_preds

y_true_val, y_pred_val, topk_val = eval_split(val_df, "Validation", topk=3)
y_true_test, y_pred_test, topk_test = eval_split(test_df, "Test", topk=3)


=== Validation ===
Top-1  -> Accuracy: 0.9103 | F1-macro: 0.9026 | F1-weighted: 0.9094
Top-3 -> Accuracy: 0.8690

Classification report (Top-1):

                precision    recall  f1-score   support

            A1     1.0000    0.8889    0.9412         9
            A3     0.7143    0.6250    0.6667         8
            A4     0.7778    0.8750    0.8235         8
A4 med sen fas     0.8571    0.7500    0.8000         8
           B10     1.0000    1.0000    1.0000         7
            B2     0.9091    1.0000    0.9524        10
            B3     0.9697    0.8889    0.9275        36
          BBA1     1.0000    1.0000    1.0000         7
            L5     0.9091    1.0000    0.9524        10
            L7     0.8182    1.0000    0.9000         9
            N1     1.0000    1.0000    1.0000        10
           N14     0.8750    0.8750    0.8750         8
            N4     0.8889    1.0000    0.9412         8
            P2     0.8571    0.8571    0.8571         7

      accur

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

def save_eval_artifacts(df_split: pd.DataFrame,
                        name: str,
                        y_pred: list,
                        topk_preds: list,
                        output_dir: str = CFG["output_dir"]):
    """
    Saves:
      - predictions_<name>.csv  with columns: [text, true_label, pred_top1, top3, hit_top3]
      - confusion_matrix_<name>.png  confusion matrix with counts overlaid
    """
    os.makedirs(output_dir, exist_ok=True)

    texts   = df_split[CFG["text_col"]].tolist()
    y_true  = df_split[CFG["label_col"]].tolist()
    top3_as_str = [" | ".join(cands) if isinstance(cands, (list, tuple)) else str(cands) for cands in topk_preds]
    hit_top3 = [(t in cands) if isinstance(cands, (list, tuple)) else False for t, cands in zip(y_true, topk_preds)]

    df_out = pd.DataFrame({
        "text": texts,
        "true_label": y_true,
        "pred_top1": y_pred,
        "top3": top3_as_str,
        "hit_top3": hit_top3,
    })
    csv_path = os.path.join(output_dir, f"predictions_{name.lower()}.csv")
    df_out.to_csv(csv_path, index=False, encoding="utf-8-sig")
    labels = label_list
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    fig_w = max(8, len(labels) * 0.5)
    fig_h = max(6, len(labels) * 0.5)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(f"Confusion matrix ({name})")
    ax.set_xticks(np.arange(len(labels)), labels=labels, rotation=90)
    ax.set_yticks(np.arange(len(labels)), labels=labels)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)

    plt.tight_layout()
    png_path = os.path.join(output_dir, f"confusion_matrix_{name.lower()}.png")
    plt.savefig(png_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved:\n  {csv_path}\n  {png_path}")

save_eval_artifacts(val_df,  "Validation", y_pred_val,  topk_val,  output_dir=CFG["output_dir"])
save_eval_artifacts(test_df, "Test",        y_pred_test, topk_test, output_dir=CFG["output_dir"])


Saved:
  gptsw3_radiology_cls_full/predictions_validation.csv
  gptsw3_radiology_cls_full/confusion_matrix_validation.png
Saved:
  gptsw3_radiology_cls_full/predictions_test.csv
  gptsw3_radiology_cls_full/confusion_matrix_test.png


In [ ]:
import os, json
import pandas as pd

MODEL_DIR = "gptsw3_radiology_cls_full"

labels_path = os.path.join(MODEL_DIR, "labels.json")

def _save_labels_json(label_list, path=labels_path, min_samples_per_class=None):
    label_list = sorted(sorted(set(label_list)), key=lambda x: x.lower())
    meta = {
        "label_list": label_list,
        "label_to_id": {lbl: i for i, lbl in enumerate(label_list)},
        "id_to_label": {i: lbl for i, lbl in enumerate(label_list)},
        "min_samples_per_class": min_samples_per_class,
        "prompt_note": "Svar ENDAST med etiketten.",
    }
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    print(f"✅ Wrote {path} with {len(label_list)} labels.")
    return meta

label_list_source = None
label_list_built = None
min_samples = None

try:
    if "label_list" in globals() and isinstance(label_list, (list, tuple)) and len(label_list) > 0:
        label_list_built = list(label_list)
        label_list_source = "in-memory (label_list from training)"
        min_samples = CFG.get("min_samples_per_class", None) if "CFG" in globals() else None
    elif "keep_labels" in globals() and isinstance(keep_labels, (list, tuple)) and len(keep_labels) > 0:
        label_list_built = list(keep_labels)
        label_list_source = "in-memory (keep_labels)"
        min_samples = CFG.get("min_samples_per_class", None) if "CFG" in globals() else None
    else:
        if "CFG" in globals():
            excel_path = CFG["excel_path"]
            text_col = CFG["text_col"]
            label_col = CFG["label_col"]
            min_samples = CFG.get("min_samples_per_class", 50)
        else:
            excel_path = "/content/radiologue dataset.xlsx"
            text_col = "Summa"
            label_col = "End Result"
            min_samples = 50

        print(f"Reading labels from: {excel_path}")
        df_tmp = pd.read_excel(excel_path)
        df_tmp = df_tmp[[text_col, label_col]].dropna().copy()
        df_tmp[text_col] = df_tmp[text_col].astype(str).str.strip()
        df_tmp[label_col] = df_tmp[label_col].astype(str).str.strip()
        vc = df_tmp[label_col].value_counts()
        label_list_built = vc[vc >= min_samples].index.tolist()
        label_list_source = f"Excel (min_samples_per_class={min_samples})"
except Exception as e:
    raise RuntimeError(f"Could not rebuild labels.json automatically: {e}")

if not label_list_built:
    raise RuntimeError("No labels found. Ensure you still have training variables in memory or set CFG['excel_path'] correctly.")

print(f"Label source: {label_list_source}")
_save_labels_json(label_list_built, labels_path, min_samples_per_class=min_samples)

# Optional: quick peek
with open(labels_path, "r", encoding="utf-8") as f:
    meta = json.load(f)
print(f"Labels preview (first 20): {meta['label_list'][:20]}")

Label source: in-memory (label_list from training)
✅ Wrote gptsw3_radiology_cls_full/labels.json with 14 labels.
Labels preview (first 20): ['A1', 'A3', 'A4', 'A4 med sen fas', 'B10', 'B2', 'B3', 'BBA1', 'L5', 'L7', 'N1', 'N14', 'N4', 'P2']


In [ ]:
import os, json, re, torch
from typing import List, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_DIR = "gptsw3_radiology_cls_full"

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if use_bf16 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    dtype=DTYPE,
    attn_implementation="sdpa",   # portable and fast; avoids FlashAttention2 dependency
    device_map="auto",
)
model.eval()
model.config.use_cache = False

labels_json = os.path.join(MODEL_DIR, "labels.json")
if not os.path.exists(labels_json):
    raise FileNotFoundError(f"labels.json not found in {MODEL_DIR}. "
                            "Run training/export cells or copy labels.json next to the model.")

with open(labels_json, "r", encoding="utf-8") as f:
    meta = json.load(f)
label_list = meta["label_list"]

def build_label_block(labels: List[str]) -> str:
    return " | ".join(labels)
LABEL_BLOCK = build_label_block(label_list)

SWEDISH_SYSTEM_INSTR = (
    "Klassificera den kliniska texten till exakt en etikett från listan över radiologiska protokoll (End Result).\n"
    "Svara ENDAST med etiketten, inget annat."
)

def make_prompt(clinical_text: str) -> str:
    return (
        f"{SWEDISH_SYSTEM_INSTR}\n\n"
        f"Tillåtna etiketter: {LABEL_BLOCK}\n\n"
        f"Text: {clinical_text.strip()}\n\n"
        f"Svar:"
    )

def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip().lower())

norm_label_map = {normalize(lbl): lbl for lbl in label_list}

def closest_label(pred: str) -> Optional[str]:
    p = normalize(pred)
    if p in norm_label_map:
        return norm_label_map[p]
    for lbl in label_list:
        n = normalize(lbl)
        if p.startswith(n) or n.startswith(p):
            return lbl
    try:
        import difflib
        match = difflib.get_close_matches(p, list(norm_label_map.keys()), n=1, cutoff=0.0)
        if match:
            return norm_label_map[match[0]]
    except Exception:
        pass
    return None

def _extract_first_label_from_text(decoded: str) -> str:
    after = decoded.split("Svar:")[-1].strip()
    return re.split(r"[\n\r]|[.，,;:!?\u3002]", after, maxsplit=1)[0].strip()

@torch.no_grad()
def classify_one(text: str, max_length: int = 512, max_new_tokens: int = 8) -> str:
    """Top-1 label using greedy (or 1-beam) decoding."""
    prompt = make_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).to(model.device)
    gen = model.generate(
        **inputs,
        do_sample=False,
        num_beams=1,
        num_return_sequences=1,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    full = tokenizer.decode(gen[0], skip_special_tokens=True)
    pred_raw = _extract_first_label_from_text(full)
    return closest_label(pred_raw) or pred_raw

@torch.no_grad()
def classify_topk(text: str, k: int = 3, max_length: int = 512, max_new_tokens: int = 8) -> List[str]:
    """Top-k labels via beam search (deterministic, no sampling)."""
    beams = max(k, 2)
    prompt = make_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).to(model.device)
    gen = model.generate(
        **inputs,
        do_sample=False,
        num_beams=beams,
        num_return_sequences=min(k, beams),
        max_new_tokens=max_new_tokens,
        early_stopping=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    cand, seen = [], set()
    for seq in gen:
        full = tokenizer.decode(seq, skip_special_tokens=True)
        pred_raw = _extract_first_label_from_text(full)
        mapped = closest_label(pred_raw) or pred_raw
        if mapped in norm_label_map.values() and mapped not in seen:
            cand.append(mapped); seen.add(mapped)
        if len(cand) >= k:
            break
    return cand

example_text = "Anamnes: Bästa kollegor! Man som gjort en TEVAR-behandling p.g.a. trauma. Tacksam för CT av aorta i thorax för stentkontroll.   Frågeställning: "
print("Top-1:", classify_one(example_text))
print("Top-3:", classify_topk(example_text, k=3))


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Top-1: A1
Top-3: ['A1', 'A4']
